## Khai báo thư viện và định nghĩa đường dẫn

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

# --- CẤU HÌNH ---
FILE_PATH = '/content/train.txt'
TEST_PATH = '/content/test.txt'

# Định nghĩa khoảng thời gian bão (Vùng dữ liệu bị mất)
STORM_START = '1995-08-01 14:52:02'
STORM_END   = '1995-08-03 04:36:13'

print("Đã khai báo thư viện và đường dẫn thành công.")

Đã khai báo thư viện và đường dẫn thành công.


## Xây dựng hàm đọc và parsing dữ liệu thô

In [2]:
# --- CELL 2: RAW DATA PARSER ---
def load_raw_data(file_path):
    print(f"🚀 Đang đọc file: {file_path} ...")

    log_pattern = re.compile(r'(?P<host>\S+) - - \[(?P<time>.*?)\] "(?P<request>.*?)" (?P<status>\d+) (?P<bytes>\S+)')
    data = []

    try:
        with open(file_path, 'r', encoding='latin-1', errors='ignore') as f:
            for line in f:
                match = log_pattern.match(line)
                if match:
                    data.append(match.groupdict())
    except FileNotFoundError:
        print(f"❌ Lỗi: Không tìm thấy file {file_path}")
        return pd.DataFrame()

    df = pd.DataFrame(data)
    if df.empty: return df

    # Chuyển đổi kiểu dữ liệu cơ bản
    df['bytes'] = df['bytes'].replace('-', '0').astype(int)
    df['status'] = df['status'].astype(int)

    # Parse thời gian
    df['timestamp'] = pd.to_datetime(df['time'], format='%d/%b/%Y:%H:%M:%S %z')
    df.drop(columns=['time'], inplace=True)

    # Sắp xếp theo thời gian để chuẩn bị cho các bước sau
    df = df.sort_values('timestamp')

    print(f"✅ Đã parse xong {len(df)} dòng log gốc.")
    return df

## Xử lý phần dữ liệu chết

In [3]:
# --- CELL 3: TRAIN DATA PROCESSING (SPECIAL LOGIC) ---
def process_train_data(df, storm_start, storm_end):
    print("\n🔄 Đang xử lý tập TRAIN (Resample 1T + Fill 0 vùng bão)...")

    # 1. Set index để Resample
    df = df.set_index('timestamp')

    # 2. Resample 1 Phút (Tạo trục thời gian liên tục)
    # Logic:
    # - request: count (Trong vùng bão ko có log -> count = 0 -> ĐÚNG YÊU CẦU)
    # - bytes: sum (Trong vùng bão ko có log -> sum = 0 (sau khi fillna))
    df_res = df.resample('1T').agg({
        'request': 'count',
        'bytes': 'sum',
        'status': 'max' # Lấy status đại diện
    })

    # 3. Fill 0 cho các giá trị NaN (đặc biệt trong vùng bão)
    df_res = df_res.fillna(0)

    # 4. Tạo biến đặc trưng is_instorm
    df_res['is_instorm'] = 0
    # Chỉ gán = 1 trong đúng vùng bão
    df_res.loc[storm_start:storm_end, 'is_instorm'] = 1

    # 5. KHÔI PHỤC CỘT TIMESTAMP
    # Reset index để timestamp trở thành một cột bình thường
    df_res = df_res.reset_index()
    if 'index' in df_res.columns:
        df_res.rename(columns={'index': 'timestamp'}, inplace=True)

    print(f"⚡ Đã xử lý xong vùng bão ({storm_start} -> {storm_end}).")
    print(f"✅ Tổng số dòng tập Train sau khi xử lý: {len(df_res)}")
    return df_res

## Thực thi

In [4]:
# --- CELL 4: EXECUTION ---

# === 1. XỬ LÝ TẬP TRAIN ===
print("--- 1. WORKING ON TRAIN DATA ---")
df_raw_train = load_raw_data(FILE_PATH)
df_raw_train1 = df_raw_train.reset_index(drop=True)
print("Xuất file data để train model")
df_raw_train1.to_parquet('train_raw.parquet')
if not df_raw_train.empty:
    # Áp dụng logic đặc biệt cho Train
    df_train_final = process_train_data(df_raw_train, STORM_START, STORM_END)

    # Lưu file
    df_train_final.to_parquet('train_final.parquet')
    print("💾 Đã lưu file: train_final.parquet (Time Series Format)")

    # Kiểm tra nhanh
    print("\n[Preview Train Data inside Storm]")
    display(df_train_final[df_train_final['is_instorm'] == 1].head(3))


# === 2. XỬ LÝ TẬP TEST ===
print("\n--- 2. WORKING ON TEST DATA ---")
df_raw_test = load_raw_data(TEST_PATH)

if not df_raw_test.empty:
    # Với Test: CHỈ PARSING, KHÔNG LÀM GÌ KHÁC
    # Giữ nguyên cấu trúc cột (host, request, status, bytes, timestamp)
    # Không resample, không thêm is_instorm

    df_test_final = df_raw_test.reset_index(drop=True) # Reset index cho sạch

    # Lưu file
    df_test_final.to_parquet('test_final.parquet')
    print("💾 Đã lưu file: test_final.parquet (Raw Log Format)")

    # Kiểm tra nhanh
    print("\n[Preview Test Data]")
    display(df_test_final.head(3))

--- 1. WORKING ON TRAIN DATA ---
🚀 Đang đọc file: /content/train.txt ...
✅ Đã parse xong 2934961 dòng log gốc.
Xuất file data để train model

🔄 Đang xử lý tập TRAIN (Resample 1T + Fill 0 vùng bão)...


/tmp/ipython-input-197810917.py:12: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_res = df.resample('1T').agg({


⚡ Đã xử lý xong vùng bão (1995-08-01 14:52:02 -> 1995-08-03 04:36:13).
✅ Tổng số dòng tập Train sau khi xử lý: 76320
💾 Đã lưu file: train_final.parquet (Time Series Format)

[Preview Train Data inside Storm]


,timestamp,request,bytes,status,is_instorm
45533,1995-08-01 14:53:00-04:00,0,0,0.0,1
45534,1995-08-01 14:54:00-04:00,0,0,0.0,1
45535,1995-08-01 14:55:00-04:00,0,0,0.0,1



--- 2. WORKING ON TEST DATA ---
🚀 Đang đọc file: /content/test.txt ...
✅ Đã parse xong 526651 dòng log gốc.
💾 Đã lưu file: test_final.parquet (Raw Log Format)

[Preview Test Data]


,host,request,status,bytes,timestamp
0,ix-mia1-02.ix.netcom.com,GET /ksc.html HTTP/1.0,200,7087,1995-08-23 00:00:00-04:00
1,internet-gw.watson.ibm.com,GET /history/apollo/pad-abort-test-2/pad-abort...,200,1292,1995-08-23 00:00:05-04:00
2,ix-mia1-02.ix.netcom.com,GET /images/ksclogo-medium.gif HTTP/1.0,200,5866,1995-08-23 00:00:06-04:00


In [ ]:
display(df_raw_train1.head(3))

,host,request,status,bytes,timestamp
0,199.72.81.55,GET /history/apollo/ HTTP/1.0,200,6245,1995-07-01 00:00:01-04:00
1,unicomp6.unicomp.net,GET /shuttle/countdown/ HTTP/1.0,200,3985,1995-07-01 00:00:06-04:00
2,199.120.110.21,GET /shuttle/missions/sts-73/mission-sts-73.ht...,200,4085,1995-07-01 00:00:09-04:00
